# 12. Korrelation

Förra notebooken handlade om samband mellan två *kvalitativa* variabler. Nu tar vi två **kvantitativa**:
hänger studietid ihop med provresultat? Längd med vikt? Pris med efterfrågan?

Sambandet *visas* med ett spridningsdiagram och *mäts* med korrelationskoefficienten r.

**Efter den här notebooken kan du**
- läsa ett spridningsdiagram: riktning, form och styrka
- räkna ut och tolka Pearsons korrelationskoefficient r
- förklara vad r *inte* fångar: icke-linjära samband och uteliggare
- testa om en korrelation är statistiskt säkerställd
- förklara varför korrelation inte är kausalitet, och vad en bakomliggande variabel är

**Förkunskaper:** [8. Hypotesprövning och p-värden](../08_hypotesprovning/08_hypotesprovning.ipynb)
och z-värden från [4. Fördelningar](../04_fordelningar/04_fordelningar.ipynb)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)

## 12.1 Spridningsdiagrammet

50 studenter har uppgett hur många timmar de pluggade inför ett prov. Varje punkt är en student.

In [ ]:
study_hours = rng.uniform(0, 20, 50).round(1)
exam_score = np.clip(40 + 2.5 * study_hours + rng.normal(0, 8, 50), 0, 100).round()

plt.scatter(study_hours, exam_score)
plt.xlabel("Studietimmar")
plt.ylabel("Provpoäng")
plt.show()

Tre saker att titta efter i ett spridningsdiagram:

1. **Riktning:** stiger punktsvärmen (positivt samband) eller faller den (negativt)?
2. **Form:** följer den en rät linje, en kurva eller inget mönster alls?
3. **Styrka:** ligger punkterna tätt samlade kring mönstret eller utspridda?

Här: positivt, ungefär linjärt och starkt.

## 12.2 Korrelationskoefficienten r

**Pearsons korrelationskoefficient r** sammanfattar riktning och styrka hos ett *linjärt* samband i ett tal mellan −1 och 1:

- r = 1: alla punkter på en stigande rät linje
- r = 0: inget linjärt samband
- r = −1: alla punkter på en fallande rät linje

In [ ]:
r, p_value = stats.pearsonr(study_hours, exam_score)
print(f"r = {r:.2f}")

### Hur räknas r ut?

Idén är enkel när man har z-värden. Räkna om båda variablerna till z-värden. En student som pluggat mer än medel har
positivt z för timmar. Har hen också fått mer än medel i poäng är även det z-värdet positivt, och **produkten** blir positiv.
Två negativa z-värden ger också en positiv produkt. Ett av varje ger en negativ.

r är (i stort sett) medelvärdet av dessa produkter:

$$r = \frac{1}{n-1}\sum z_x \cdot z_y$$

In [ ]:
z_x = (study_hours - study_hours.mean()) / study_hours.std(ddof=1)
z_y = (exam_score - exam_score.mean()) / exam_score.std(ddof=1)

steps = pd.DataFrame({"hours": study_hours, "score": exam_score, "z_x": z_x, "z_y": z_y, "product": z_x * z_y})
display(steps.head(8).round(2))

print(f"Andel positiva produkter: {(steps['product'] > 0).mean():.0%}")
print(f"r för hand:  {steps['product'].sum() / (len(steps) - 1):.4f}")
print(f"r med scipy: {r:.4f}")

De flesta produkterna är positiva: de som ligger över medel i timmar ligger oftast över medel i poäng. Därför blir r positivt.

Eftersom r bygger på z-värden spelar **enheterna ingen roll**. r blir detsamma om studietiden mäts i minuter i stället för timmar.

## 12.3 Få en känsla för r

Hur ser r = 0,3 ut jämfört med r = 0,8? Cellen nedan skapar data med valfri mängd brus.

🔮 **Gissa först:** titta på diagrammet innan du läser rubriken. Vilket r skulle du gissa på?

In [ ]:
noise = 1.0         # ← ändra: pröva 0, 0.3, 1, 3 och 10
slope = 1.0         # ← ändra: pröva -1 och 5

x = rng.normal(0, 1, 200)
y = slope * x + rng.normal(0, noise, 200)

plt.scatter(x, y, s=10)
plt.xlabel("x")
plt.ylabel("y")
plt.title(f"r = {stats.pearsonr(x, y)[0]:.2f}")
plt.show()

🧪 **Labba:**
1. Variera `noise`. Hur ser r ≈ 0,3 ut? r ≈ 0,7? r ≈ 0,95?
2. Sätt `slope = -1`. Vad händer med r?
3. Sätt `noise = 1` och jämför `slope = 1` med `slope = 5`. Linjen blir brantare – men vad händer med r, och varför?

<details>
<summary>💡 <b>Svar på fråga 3</b></summary>

r mäter hur **tätt** punkterna ligger kring en linje, inte hur **brant** linjen är. Med `slope = 5` är bruset litet jämfört
med lutningen, så punkterna ligger tätare kring linjen och r blir högre. Men en brant linje kan ha lågt r (mycket brus) och en
flack linje kan ha högt r (lite brus). Lutningen är ämnet för nästa notebook.

</details>

Grova riktmärken för |r|: under 0,3 svagt, 0,3–0,7 måttligt, över 0,7 starkt. Vad som räknas som starkt beror på ämnet.

## 12.4 Vad r inte ser

### Icke-linjära samband

In [ ]:
x = rng.normal(0, 1, 200)
examples = {
    "starkt positivt": x + rng.normal(0, 0.3, 200),
    "svagt positivt": x + rng.normal(0, 1.5, 200),
    "negativt": -x + rng.normal(0, 0.7, 200),
    "icke-linjärt": x**2 + rng.normal(0, 0.3, 200),
}

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, (name, y) in zip(axes, examples.items()):
    ax.scatter(x, y, s=8)
    ax.set_title(f"{name}\nr = {stats.pearsonr(x, y)[0]:.2f}")
plt.tight_layout()
plt.show()

I den sista bilden finns ett mycket tydligt samband: om du känner x kan du nästan exakt förutsäga y.
Ändå är r nära noll, eftersom sambandet inte är en rät linje.

> **r = 0 betyder "inget *linjärt* samband", inte "inget samband".** Rita alltid ett spridningsdiagram.

### Uteliggare

En enda avvikande punkt kan både skapa och förstöra en korrelation.

In [ ]:
x_base = rng.normal(50, 5, 20)
y_base = rng.normal(50, 5, 20)             # inget samband alls

outlier_x, outlier_y = 50, 50              # ← ändra: pröva (100, 100) och (100, 0)

x_all = np.append(x_base, outlier_x)
y_all = np.append(y_base, outlier_y)

plt.scatter(x_base, y_base, label="20 punkter utan samband")
plt.scatter(outlier_x, outlier_y, color="red", label="den 21:a punkten")
plt.xlabel("x")
plt.ylabel("y")
plt.title(f"r utan röd punkt = {stats.pearsonr(x_base, y_base)[0]:.2f},   r med = {stats.pearsonr(x_all, y_all)[0]:.2f}")
plt.legend()
plt.show()

🧪 **Labba:** Flytta den röda punkten till (100, 100) och sedan till (100, 0). En enda observation av 21 avgör hela resultatet.

## 12.5 Är korrelationen statistiskt säkerställd?

Även två variabler helt utan samband får ett r som inte är exakt noll i ett stickprov.
`pearsonr` ger därför också ett p-värde för $H_0$: *ingen linjär korrelation i populationen*.

In [ ]:
r, p_value = stats.pearsonr(study_hours, exam_score)
print(f"Studietimmar och provpoäng: r = {r:.2f},  p-värde = {p_value:.2g}")

p-värdet är långt under 0,05, så sambandet är statistiskt säkerställt.

Hur stort r kan slumpen ge? Det beror på stickprovets storlek. Vi drar två variabler **utan något samband** många gånger:

In [ ]:
n = 10          # ← ändra: pröva 5, 10, 30 och 300

random_r = [stats.pearsonr(rng.normal(0, 1, n), rng.normal(0, 1, n))[0] for _ in range(2000)]

plt.hist(random_r, bins=40, edgecolor="white")
plt.xlim(-1, 1)
plt.xlabel("r mellan två variabler utan samband")
plt.ylabel("Antal stickprov")
plt.title(f"n = {n}:  95 % av r-värdena ligger inom ±{np.quantile(np.abs(random_r), 0.95):.2f}")
plt.show()

🧪 **Labba:** Med n = 10 är r = 0,5 inget ovanligt av ren slump. Hur stort r kan slumpen ge med n = 300?

✏️ **Din tur:** En kollega har mätt två saker på 8 personer och fått r = 0,60. "Ett starkt samband!" säger hen.
Använd simuleringen ovan (med n = 8) för att bedöma hur imponerad du bör vara. Hur stor andel av de slumpmässiga r-värdena
är minst 0,60 i absolutbelopp?

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
random_r_8 = np.array([stats.pearsonr(rng.normal(0, 1, 8), rng.normal(0, 1, 8))[0] for _ in range(5000)])
print(f"Andel med |r| ≥ 0,60 av ren slump: {(np.abs(random_r_8) >= 0.60).mean():.1%}")
# Drygt 10 % – alltså ett p-värde över 0,05. Med 8 observationer är r = 0,60 inte statistiskt säkerställt.
```

</details>

## 12.6 Korrelation är inte kausalitet

Det mest citerade i hela statistiken, och det som oftast glöms bort.

Vi simulerar 200 sommardagar. **Temperaturen** påverkar både glassförsäljningen och antalet badolyckor.
Glass och badolyckor har ingen direkt koppling till varandra.

In [ ]:
temperature = rng.uniform(10, 30, 200)                                # °C
ice_cream = 20 + 8 * temperature + rng.normal(0, 25, 200)             # sålda glassar
swim_accidents = 0.4 * temperature + rng.normal(0, 2, 200)            # olycksindex

r_ice, p_ice = stats.pearsonr(ice_cream, swim_accidents)

plt.scatter(ice_cream, swim_accidents, s=12)
plt.xlabel("Sålda glassar")
plt.ylabel("Badolyckor (index)")
plt.title(f"r = {r_ice:.2f},  p = {p_ice:.1g}")
plt.show()

En tydlig och höggradigt signifikant korrelation. Borde vi förbjuda glass för att minska antalet badolyckor?

Naturligtvis inte. Båda beror på en **bakomliggande variabel** (på engelska *confounder*): temperaturen.
Ett sätt att se det är att jämföra dagar som är **lika varma**. Om glass orsakade olyckor borde sambandet finnas kvar då.

In [ ]:
low_temp, high_temp = 20, 22          # ← ändra: pröva (12, 14) och (26, 28)

similar_days = (temperature >= low_temp) & (temperature < high_temp)
r_within = stats.pearsonr(ice_cream[similar_days], swim_accidents[similar_days])[0]

plt.scatter(ice_cream, swim_accidents, s=12, color="lightgray", label="alla dagar")
plt.scatter(ice_cream[similar_days], swim_accidents[similar_days], s=20, color="red",
            label=f"dagar med {low_temp}–{high_temp} °C")
plt.xlabel("Sålda glassar")
plt.ylabel("Badolyckor (index)")
plt.title(f"{similar_days.sum()} dagar med liknande temperatur: r = {r_within:.2f}")
plt.legend()
plt.show()

🧪 **Labba:** Pröva några olika temperaturintervall. Bland dagar med samma temperatur försvinner sambandet nästan helt.

Det finns tre förklaringar att överväga varje gång två variabler A och B korrelerar:

| Förklaring | Exempel |
|---|---|
| **A orsakar B** | Studietid → provresultat |
| **B orsakar A** (omvänd kausalitet) | "Personer som bantar väger mer": det är vikten som leder till bantning, inte tvärtom |
| **En tredje variabel orsakar båda** | Temperatur → glass och badolyckor |

Och så en fjärde: **slumpen**, särskilt i små stickprov eller när man har letat bland många variabler.

För att belägga orsakssamband krävs i regel ett **experiment**, där man själv lottar vem som får vilken behandling.
Lottningen gör att grupperna blir lika i alla andra avseenden. Det är därför A/B-testet i notebook 9 kunde säga något om
orsak och verkan.

✏️ **Din tur:** Ingen kod. Föreslå en alternativ förklaring till varje rubrik:

**A.** "Barn med större skor läser bättre."

**B.** "Länder med fler tv-apparater per invånare har högre medellivslängd."

**C.** "Personer som går på gym regelbundet är friskare – träning förebygger sjukdom."

<details>
<summary>💡 <b>Svar</b></summary>

**A.** Bakomliggande variabel: ålder. Äldre barn har större fötter och läser bättre.

**B.** Bakomliggande variabel: välstånd. Rika länder har både fler tv-apparater och bättre sjukvård, kost och sanitet.

**C.** Träning kan mycket väl vara bra, men korrelationen visar det inte. Omvänd kausalitet: den som är sjuk går inte på gym.
Bakomliggande variabler: inkomst, ålder och livsstil i övrigt.

</details>

## Sammanfattning

- Börja alltid med ett **spridningsdiagram**: riktning, form, styrka.
- **r** mäter riktning och styrka hos ett **linjärt** samband, mellan −1 och 1, oberoende av enheter.
- r säger inget om hur *brant* sambandet är, missar icke-linjära samband och är känsligt för uteliggare.
- I små stickprov kan slumpen ge stora r. Titta på p-värdet.
- **Korrelation är inte kausalitet.** Fråga efter omvänd kausalitet och bakomliggande variabler.

### Kontrollfrågor
1. Vad händer med r om alla x-värden multipliceras med 100? Om alla y-värden byter tecken?
2. r mellan ålder och reaktionstid är 0,02 i ett stort stickprov. Kan man dra slutsatsen att ålder inte påverkar reaktionstiden?
3. Varför kan ett lottat experiment visa orsakssamband när en observationsstudie inte kan det?

<details>
<summary>💡 <b>Svar</b></summary>

1. Ingenting i det första fallet (r beror inte på enheten). I det andra byter r tecken.
2. Nej, bara att det inte finns något *linjärt* samband. Reaktionstiden kan till exempel vara bäst hos unga vuxna och sämre
   hos både barn och äldre, vilket ger ett U-format samband med r nära 0.
3. Lottningen gör att grupperna i genomsnitt är lika i alla bakomliggande variabler. Då återstår bara behandlingen som förklaring
   till en skillnad (utöver slumpen, som testet hanterar).

</details>

**Nästa notebook:** [13. Enkel linjär regression](../13_linjar_regression/13_linjar_regression.ipynb)